# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratham6306/ml-internship-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb huggingface_hub pyarrow

In [3]:
import os
import duckdb
import pandas as pd
from huggingface_hub import login, hf_hub_download

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(HF_TOKEN)

In [5]:
from huggingface_hub import hf_hub_download

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

In [6]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(march_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [7]:
import duckdb

con = duckdb.connect()

df = con.execute(f"""
SELECT *
FROM read_parquet('{march_path}')
""").df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [8]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

A page should be prioritized for CTR optimization if it receives a high number of Google Search impressions but relatively few clicks. High impressions indicate strong visibility, while a low click count suggests users are not selecting the page in search results.

### Reason Code

- **ctr_optimization** – High search visibility but comparatively low clicks.
- **not_prioritized** – Does not meet the baseline rule.

In [9]:
import pandas as pd
import numpy as np

# ---------------------------------------
# Use only rows where GSC data exists
# ---------------------------------------
baseline_df = df[df["gsc_data_available"]].copy()

# ---------------------------------------
# Calculate CTR
# ---------------------------------------
baseline_df["ctr"] = np.where(
    baseline_df["gsc_impressions"] > 0,
    baseline_df["gsc_clicks"] / baseline_df["gsc_impressions"],
    0
)

# ---------------------------------------
# Define baseline rule
# ---------------------------------------
HIGH_IMPRESSIONS = 500
LOW_CTR = 0.03

baseline_df["high_impressions"] = (
    baseline_df["gsc_impressions"] >= HIGH_IMPRESSIONS
)

baseline_df["low_ctr"] = (
    baseline_df["ctr"] <= LOW_CTR
)

print("Baseline Rule")
print("-" * 50)
print(
    f"Prioritize pages with at least {HIGH_IMPRESSIONS} impressions "
    f"and CTR less than or equal to {LOW_CTR:.0%}."
)

print("\nReason Codes")
print("-" * 50)
print("ctr_optimization : High impressions but low CTR")
print("not_prioritized  : Rule not satisfied")

Baseline Rule
--------------------------------------------------
Prioritize pages with at least 500 impressions and CTR less than or equal to 3%.

Reason Codes
--------------------------------------------------
ctr_optimization : High impressions but low CTR
not_prioritized  : Rule not satisfied


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import os
import numpy as np

# ---------------------------------------
# Build a transparent baseline score
# ---------------------------------------

baseline_df["score"] = np.where(
    baseline_df["high_impressions"] & baseline_df["low_ctr"],
    baseline_df["gsc_impressions"],
    0
)

# ---------------------------------------
# Reason Codes
# ---------------------------------------

baseline_df["reason_code"] = np.where(
    baseline_df["score"] > 0,
    "ctr_optimization",
    "not_prioritized"
)

# ---------------------------------------
# Action Labels
# ---------------------------------------

baseline_df["action_label"] = np.where(
    baseline_df["score"] > 0,
    "Optimize CTR",
    "No Action"
)

# ---------------------------------------
# Rank highest score first
# ---------------------------------------

ranked_df = (
    baseline_df
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

# ---------------------------------------
# Keep useful columns
# ---------------------------------------

output_df = ranked_df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "score",
        "reason_code",
        "action_label"
    ]
]

# ---------------------------------------
# Write CSV
# ---------------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

output_df.to_csv(output_path, index=False)

print(f"CSV written successfully to:\n{output_path}")

print("\nTop 10 Ranked Pages\n")
display(output_df.head(10))

CSV written successfully to:
work/outputs/baseline_action_score.csv

Top 10 Ranked Pages



,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code,action_label
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.000025,40084,ctr_optimization,Optimize CTR
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,39305,252,0.006411,39305,ctr_optimization,Optimize CTR
2,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,0.000051,39003,ctr_optimization,Optimize CTR
3,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,38436,271,0.007051,38436,ctr_optimization,Optimize CTR
4,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.000000,37368,ctr_optimization,Optimize CTR
5,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,35404,225,0.006355,35404,ctr_optimization,Optimize CTR
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,34817,223,0.006405,34817,ctr_optimization,Optimize CTR
7,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,34606,235,0.006791,34606,ctr_optimization,Optimize CTR
8,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,33571,215,0.006404,33571,ctr_optimization,Optimize CTR
9,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.000000,33383,ctr_optimization,Optimize CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# ---------------------------------------
# Top 20 Review
# ---------------------------------------

top20 = ranked_df.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] > 1000,
    "High confidence",
    "Medium confidence"
)

top20["what_would_make_it_wrong"] = (
    "CTR may already be acceptable because of branded searches, "
    "seasonality, or incomplete data."
)

review = top20[
    [
        "report_date",
        "content_hash_id",
        "action_label",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print("Top 20 Review")
display(review)

print("\nIndividual Review\n")

for i, row in review.iterrows():
    print(f"{i+1}.")
    print(f"Action               : {row['action_label']}")
    print(f"Reason Code          : {row['reason_code']}")
    print(f"Confidence           : {row['confidence_note']}")
    print(f"What would make it wrong?")
    print(f"  {row['what_would_make_it_wrong']}")
    print("-" * 70)

Top 20 Review


,report_date,content_hash_id,action_label,reason_code,score,confidence_note,what_would_make_it_wrong
0,2026-03-28,content_44f34c0a90047651,Optimize CTR,ctr_optimization,40084,High confidence,CTR may already be acceptable because of brand...
1,2026-03-29,content_eadb33b5df496f4a,Optimize CTR,ctr_optimization,39305,High confidence,CTR may already be acceptable because of brand...
2,2026-03-04,content_34a70fea29d15f24,Optimize CTR,ctr_optimization,39003,High confidence,CTR may already be acceptable because of brand...
3,2026-03-28,content_eadb33b5df496f4a,Optimize CTR,ctr_optimization,38436,High confidence,CTR may already be acceptable because of brand...
4,2026-03-04,content_945d6ff91386c817,Optimize CTR,ctr_optimization,37368,High confidence,CTR may already be acceptable because of brand...
5,2026-03-30,content_eadb33b5df496f4a,Optimize CTR,ctr_optimization,35404,High confidence,CTR may already be acceptable because of brand...
6,2026-03-27,content_eadb33b5df496f4a,Optimize CTR,ctr_optimization,34817,High confidence,CTR may already be acceptable because of brand...
7,2026-03-31,content_eadb33b5df496f4a,Optimize CTR,ctr_optimization,34606,High confidence,CTR may already be acceptable because of brand...
8,2026-03-24,content_eadb33b5df496f4a,Optimize CTR,ctr_optimization,33571,High confidence,CTR may already be acceptable because of brand...
9,2026-03-30,content_fec55986a1868d62,Optimize CTR,ctr_optimization,33383,High confidence,CTR may already be acceptable because of brand...



Individual Review

1.
Action               : Optimize CTR
Reason Code          : ctr_optimization
Confidence           : High confidence
What would make it wrong?
  CTR may already be acceptable because of branded searches, seasonality, or incomplete data.
----------------------------------------------------------------------
2.
Action               : Optimize CTR
Reason Code          : ctr_optimization
Confidence           : High confidence
What would make it wrong?
  CTR may already be acceptable because of branded searches, seasonality, or incomplete data.
----------------------------------------------------------------------
3.
Action               : Optimize CTR
Reason Code          : ctr_optimization
Confidence           : High confidence
What would make it wrong?
  CTR may already be acceptable because of branded searches, seasonality, or incomplete data.
----------------------------------------------------------------------
4.
Action               : Optimize CTR
Reason Code   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# ---------------------------------------
# Weak Picks
# ---------------------------------------

print("=" * 70)
print("Weak Picks")
print("=" * 70)

weak_picks = ranked_df[
    (ranked_df["score"] > 0) &
    (ranked_df["gsc_clicks"] > 50)
].head(10)

if weak_picks.empty:
    print("No obvious weak picks found.")
else:
    display(
        weak_picks[
            [
                "content_hash_id",
                "gsc_impressions",
                "gsc_clicks",
                "ctr",
                "score",
                "reason_code"
            ]
        ]
    )

print("\nPossible reasons these picks may be weak:")
print("- CTR may already be acceptable for that query.")
print("- Branded searches can naturally have different CTR.")
print("- Seasonal trends may temporarily affect search behavior.")
print("- Search intent may not match the page title.")

# ---------------------------------------
# Leakage Check
# ---------------------------------------

print("\n" + "=" * 70)
print("Leakage Check")
print("=" * 70)

used_columns = [
    "gsc_impressions",
    "gsc_clicks"
]

print("Columns used in baseline rule:")
for c in used_columns:
    print(f"• {c}")

print("\nChecks")

print("✓ No future-window columns used.")
print("✓ No label-derived columns used.")
print("✓ No FlyRank product flags used.")
print("✓ Baseline uses only current observed GSC metrics.")
print("✓ Rule is transparent and reproducible.")

Weak Picks


,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code
1,content_eadb33b5df496f4a,39305,252,0.006411,39305,ctr_optimization
3,content_eadb33b5df496f4a,38436,271,0.007051,38436,ctr_optimization
5,content_eadb33b5df496f4a,35404,225,0.006355,35404,ctr_optimization
6,content_eadb33b5df496f4a,34817,223,0.006405,34817,ctr_optimization
7,content_eadb33b5df496f4a,34606,235,0.006791,34606,ctr_optimization
8,content_eadb33b5df496f4a,33571,215,0.006404,33571,ctr_optimization
12,content_eadb33b5df496f4a,32665,221,0.006766,32665,ctr_optimization
13,content_eadb33b5df496f4a,32462,216,0.006654,32462,ctr_optimization
14,content_eadb33b5df496f4a,31713,214,0.006748,31713,ctr_optimization
16,content_eadb33b5df496f4a,31133,198,0.006360,31133,ctr_optimization



Possible reasons these picks may be weak:
- CTR may already be acceptable for that query.
- Branded searches can naturally have different CTR.
- Seasonal trends may temporarily affect search behavior.
- Search intent may not match the page title.

Leakage Check
Columns used in baseline rule:
• gsc_impressions
• gsc_clicks

Checks
✓ No future-window columns used.
✓ No label-derived columns used.
✓ No FlyRank product flags used.
✓ Baseline uses only current observed GSC metrics.
✓ Rule is transparent and reproducible.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.